# PC-FNO3D — servir le modèle depuis Colab

Le GPU et les poids entraînés sont ici, sur le Drive. Ce notebook ouvre le modèle à
l'extérieur : **une URL publique** qui expose l'API JSON, le serveur MCP et l'application
web. Une autre application — ou un agent — envoie alors ses requêtes directement au vrai
modèle, sans rien copier.

```
   autre app / agent  ──https──►  tunnel  ──►  Colab (GPU + poids)
                                              /api/...  /mcp  /app/
```

**Avant de commencer**

1. Sur ton poste : `python scripts/package_for_colab.py`, puis dépose
   `artifacts/pcfno3d_code.zip` à la racine de ton Drive (*MyDrive*). L'archive ne
   contient que le code, quelques centaines de kilo-octets.
2. Le Drive doit déjà contenir le dataset `aneo_dataset_3d_v2` et le dossier de run
   `aneo_fno3d_runs/v7` avec `best_dso_v7.pt` (ou `best.pt`) **et** `norm_stats.json`.
   Sans eux, le notebook s'arrête avec un message qui dit quoi faire.
3. Exécution → Modifier le type d'exécution → **GPU** (sinon ça marche, mais lentement).

**À savoir, sans détour**

- La session Colab est **éphémère** : inactivité ~90 min, 12 h au maximum. L'URL publique
  meurt avec elle, et change à chaque relance. Ce n'est pas un hébergement.
- Le tunnel donne une **URL publique** : c'est la **clé d'API** qui protège les données,
  pas le secret de l'URL. Ne mets jamais la clé dans un dépôt ni dans le salon des agents.

## 1. Drive, code, dépendances

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Installe le code depuis l'archive du Drive (ou depuis un dossier, si tu préfères).
import shutil, subprocess, sys, zipfile
from pathlib import Path

ARCHIVE = Path("/content/drive/MyDrive/pcfno3d_code.zip")
DOSSIER = Path("/content/drive/MyDrive/PFE")      # alternative : un dossier sur le Drive
CIBLE   = Path("/content/PFE")

if CIBLE.exists():
    shutil.rmtree(CIBLE)

if ARCHIVE.exists():
    CIBLE.mkdir(parents=True)
    with zipfile.ZipFile(ARCHIVE) as zf:
        zf.extractall(CIBLE)
    print(f"code extrait de {ARCHIVE.name}")
elif DOSSIER.exists():
    # copie sur le disque local : bien plus rapide que de travailler sur le Drive monté
    shutil.copytree(DOSSIER, CIBLE, ignore=shutil.ignore_patterns(
        "artifacts", "aneo_dataset_3d_v2", "data_*", "*.rar", "*.zip", ".git",
        "__pycache__", "notebooks"))
    print(f"code copié depuis {DOSSIER}")
else:
    raise FileNotFoundError(
        f"Ni {ARCHIVE} ni {DOSSIER} sur le Drive.\n"
        "Sur ton poste : python scripts/package_for_colab.py, puis dépose "
        "artifacts/pcfno3d_code.zip à la racine de MyDrive.")

%cd /content/PFE
!pip install -q -e ".[serve]" 2>&1 | tail -3
print("\ninstallation terminée")

## 2. Vérifier ce que le Drive contient

`info` liste la configuration effective et dit, pour chaque artefact, s'il est présent.
Les trois lignes qui comptent : **checkpoint DSO** (ou *best*), **statistiques**, et
**index du cache**.

In [ ]:
!python -m pcfno3d -c colab info

## 3. (facultatif) Mettre quelques instants en cache

Deux façons d'interroger le modèle :

| Requête | Besoin |
|---|---|
| un **instant du dataset** (`wind_at_point`, `/api/point`, …) | le cache de cet instant |
| un **CSV de mesures de surface** (`POST /api/reconstruct/upload`) | rien de plus |

Construire les 100 jours est long. Pour ouvrir le service tout de suite, mets en cache
deux ou trois jours de test — ça suffit pour que `list_instants` et les requêtes par point
fonctionnent. Saute cette cellule si `info` montre déjà un index de cache, ou si tu
n'envoies que des CSV.

In [ ]:
JOURS = [4, 11]        # jours de test ; [] pour ne rien construire

if JOURS:
    !python -m pcfno3d -c colab build-cache --days {",".join(map(str, JOURS))}

## 4. Ouvrir le service

Un seul appel : il vérifie les poids, crée une clé d'API, ouvre le tunnel, démarre uvicorn
et attend que `/api/health` réponde. Le premier démarrage lit la grille (127 Mo) et les
poids : compte une à deux minutes.

Le tunnel est ouvert **avant** le service, parce que le transport MCP refuse un en-tête
`Host` qui ne lui a pas été déclaré et que le nom du tunnel est tiré au hasard.

In [ ]:
import sys
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

from service.colab import serve_public

info = serve_public(config="colab", key_name="app-externe", scope="read")

## 5. Essayer, depuis le notebook

Si ces deux cellules répondent, une application extérieure obtiendra la même chose.

In [ ]:
import json, urllib.request

def appel(chemin):
    req = urllib.request.Request(info["public_url"] + chemin,
                                 headers={"X-API-Key": info["api_key"]})
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.loads(r.read().decode())

meta = appel("/api/meta")
print("modèle :", meta["modele"], "|", meta["device"])
print("grille :", meta["grille"], "| niveaux :", meta["z_levels_asl_m"])
print("domaine :", meta["domaine_m"], "m")

point = appel("/api/point?x=5000&y=5000&z=120")
print("\nvent à (5000, 5000, 120 m) :",
      round(point["wind_speed_m_s"], 2), "m/s,",
      round(point["wind_direction_deg"], 1), "° (d'où vient le vent)")

In [ ]:
# Le serveur MCP, par le même tunnel et la même clé
payload = json.dumps({"jsonrpc": "2.0", "id": 1, "method": "tools/list",
                      "params": {}}).encode()
req = urllib.request.Request(
    info["mcp_url"], data=payload,
    headers={"X-API-Key": info["api_key"], "Content-Type": "application/json",
             "Accept": "application/json, text/event-stream"})
with urllib.request.urlopen(req, timeout=120) as r:
    outils = json.loads(r.read().decode())["result"]["tools"]
print(len(outils), "outils MCP :")
for o in outils:
    print("  -", o["name"], "—", (o.get("description") or "").splitlines()[0][:70])

## 6. Ce que tu transmets à l'autre application

La cellule 4 a déjà affiché l'essentiel. Pour le retrouver :

In [ ]:
from service.colab import summary
print(summary(info))

Côté client MCP, la déclaration est celle affichée ci-dessus. Côté HTTP :

```bash
curl -H "X-API-Key: <la clé>" "<l'URL>/api/point?x=5000&y=5000&z=120"
curl -H "X-API-Key: <la clé>" "<l'URL>/api/profile?x=5000&y=5000"
curl -H "X-API-Key: <la clé>" "<l'URL>/api/field?z=120&stride=4"
curl -X POST -H "X-API-Key: <la clé>" -F "file=@surface.csv" \
     "<l'URL>/api/reconstruct/upload"
```

Le détail des trois interfaces, du format binaire et des conventions (direction météo,
altitudes ASL, `-9999` sous le relief) est dans `docs/INTEGRATION.md`.

**Trois pièges, les mêmes qu'ailleurs :** la direction dit *d'où vient* le vent ; les
altitudes sont au-dessus du niveau de la mer, pas du sol ; sous le relief, la valeur est
`null` en JSON et `-9999.0` en binaire — ne jamais moyenner sans les écarter.

## 7. Surveiller, puis arrêter

In [ ]:
from service.colab import tail_log
print(tail_log(info, 30))

In [ ]:
# Garde la session vivante tant que l'autre application travaille.
# Interromps la cellule (le carré) pour reprendre la main : le service continue.
import time
while True:
    time.sleep(60)
    if info["process"].poll() is not None:
        print("le service s'est arrêté — voir tail_log(info)")
        break
    if info["tunnel"] and not info["tunnel"].alive:
        print("le tunnel est tombé — relance la cellule 4")
        break

In [ ]:
from service.colab import stop
stop(info)